In [1]:
import numpy as np 
import pandas as pd
from src import utils, redcells, db
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output
from pathlib import Path
%matplotlib inline
%load_ext autoreload
%autoreload 2
ROOT_PATH = "Z:/data/PROC"
RET_PATH = "D:/retinotopy/aligned_xy/behav"
MDL_PATH = "C:/Users/labadmin/Documents/category-neural/data"

In [2]:
db = db.get_sessions()
db = db.query("recording == True & obj == 'old'").reset_index(drop=True)
db

,mname,datexp,blk,session,round_id,recording,obj
0,VG11,2024_10_15,4,all rewarded before,1,True,old
1,VG11,2024_10_16,2,first training,1,True,old
2,VG11,2024_10_31,2,last training,1,True,old
3,VG11,2024_11_01,2,all rewarded after,1,True,old
4,VG11,2024_11_04,2,all rewarded before,2,True,old
5,VG11,2024_11_05,3,first training,2,True,old
6,VG11,2024_11_14,2,last training,2,True,old
7,VG11,2024_11_15,2,all rewarded after,2,True,old
8,VG14,2024_10_15,2,all rewarded before,1,True,old
9,VG14,2024_10_16,2,first training,1,True,old


In [21]:
for i, row in db.iterrows():
    name, datexp, blk = row['mname'], row["datexp"], row["blk"]    
    m = utils.load_mouse(name, datexp, blk, load_neurons=True, interp_behav=True, load_retinotopy=True, return_iscell=True,
                        data_path=ROOT_PATH, ret_path=RET_PATH,  mdl_path=MDL_PATH, opt_tlag = 1)
    frameselector = utils.get_frameselector(m)
    istim_df = pd.DataFrame(m._timeline['istim'], columns=['istim'])
    istim_df["trial_no"] = np.arange(1, len(istim_df)+1).astype(float)
    m.frameselector = (frameselector.reset_index()
                    .rename(columns = {'index':'frame'})
                    .merge(istim_df, left_on='trial_no', right_on='trial_no', how='left')
                    .set_index('frame'))
    green_channel = Path(ROOT_PATH).joinpath(name, datexp, blk, "suite2p")
    m.isred = redcells.get_redcells(green_channel)
    condition = (m._snr>=.25) & (m._is_cell[:,0].astype(bool))
    m.isred = m.isred[condition]
    m._spks = m._spks[condition]
    m._xpos = m._xpos[condition]
    m._ypos = m._ypos[condition]
    m._iplane = m._iplane[condition]
    m.iarea = m.iarea[condition]
    m.iregion = m.iregion[condition]
    m.xy_t = m.xy_t[condition]
    m._snr = m._snr[condition]
    m.trial_dict = utils.get_trialno_bytype(m.frameselector) #trial_no dict
    # lets get the interpolated spks
    m.interp_spks = utils.interp_spks_by_corridorlength(m, m.frameselector, z = True, corridor_length=400)
    save_path = r"D:\mouseobj" 
    utils.compute_dprime(m, discrimination_region = (0,100), corridor_length = 400, nogray = False)
    utils.save_mouse(m, compressed=False, mdl_path=save_path)
    clear_output(wait=True)

Checking if model object exists ...
Timeline with fname: Timeline_VG34_2025_10_10_3.mat not found, trying with fname: VG34_2025_10_10_3.mat 
Timeline file is in v7.3 format, loading with h5py
number of recording planes: 20
planes: 20


100%|██████████| 20/20 [06:23<00:00, 19.16s/it]


Loading retinotopy data from D:\retinotopy\aligned_xy\behav\VG34_2025_10_10_3_behav.npz
nrecording_rois: 20
(25805, 2)
45 12555
interpolating 19935 neurons, 12511 frames to
the vector of distance with shape: (12511,)
neurons: 19935, trials: 399, corridor length: 400
dprime saved in MouseObject.train_dp (neurons) using even trials
Mouse object saved to D:\mouseobj\VG34\2025_10_10\3
